# Preprocessing & Model Training - Health Risk Prediction

Training ML models to predict 7 health risks from self-reported features.
This is for the AI Health Coach project.

Steps:
1. Load & clean data
2. Feature engineering
3. Encode categorical features
4. Train-test split
5. Try different models, pick the best
6. Evaluate on all 7 targets
7. Save everything for the app

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, json, os, time
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, auc)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 120

## 1. Load & clean

In [ ]:
df = pd.read_csv("synthetic_health_risk_75k_v1.csv", keep_default_na=False, na_values=[])
print(df.shape)
df.head(3)

In [ ]:
# renaming Omnivore -> Non Vegetarian (makes more sense for our use case)
df['diet_type'] = df['diet_type'].replace('Omnivore', 'Non Vegetarian')
print(df['diet_type'].value_counts())

In [ ]:
# separate features from targets
risk_score_cols = [c for c in df.columns if 'risk_score' in c or c == 'general_physical_health_score']
risk_level_cols = [c for c in df.columns if 'risk_level' in c or c == 'general_physical_health_level']

print(f"Features: {df.shape[1] - len(risk_score_cols) - len(risk_level_cols) - 1}")  # -1 for bmi_category
print(f"Target columns: {len(risk_level_cols)} levels + {len(risk_score_cols)} scores")

## 2. Feature engineering

Adding some derived features that might help the model pick up patterns.

In [ ]:
df_proc = df.copy()

# bmi risk bucket (WHO thresholds)
df_proc['bmi_risk_cat'] = pd.cut(df_proc['bmi'],
    bins=[0, 18.5, 25, 30, 35, 40, 100], labels=[0,1,2,3,4,5]).astype(int)

# how far from ideal sleep (7.5h) - both too little and too much is bad
df_proc['sleep_deviation'] = np.abs(df_proc['avg_sleep_hours'] - 7.5)

# count of diabetes warning signs
df_proc['diabetes_symptom_count'] = (
    (df_proc['frequent_urination'] == 'Yes').astype(int) +
    (df_proc['slow_wound_healing'] == 'Yes').astype(int) +
    (df_proc['numbness_tingling'] == 'Yes').astype(int) +
    (df_proc['perceived_appetite'] == 'Excessive').astype(int)
)

# total family history burden
fh_cols = ['family_history_diabetes', 'family_history_heart_disease',
           'family_history_hypertension', 'family_history_obesity']
df_proc['family_history_load'] = sum((df_proc[c] == 'Yes').astype(int) for c in fh_cols)

# combined stress score
df_proc['stress_anxiety_composite'] = (
    df_proc['stress_level'] + df_proc['anxiety_level'] + df_proc['work_stress']) / 3

# diet quality score (higher = healthier)
df_proc['healthy_diet_score'] = (
    (df_proc['eat_fruits_daily'] == 'Yes').astype(int) +
    (df_proc['eat_veggies_daily'] == 'Yes').astype(int) +
    (df_proc['eat_processed_food'].isin(['Never', 'Rarely'])).astype(int) +
    (df_proc['diet_type'].isin(['Mediterranean', 'Vegetarian', 'Vegan'])).astype(int)
)

# bmi gets worse with age
df_proc['bmi_age_interaction'] = df_proc['bmi'] * np.log1p(df_proc['age'])

# sedentary + high screen time combo
ex_map = {'Sedentary': 0, 'Light': 1, 'Moderate': 2, 'Active': 3}
df_proc['exercise_num'] = df_proc['exercise_level'].map(ex_map)
df_proc['sedentary_screen_idx'] = (3 - df_proc['exercise_num']) * df_proc['screen_time_hours'] / 3

# flag for women of reproductive age (relevant for PCOS/menstrual features)
df_proc['is_female_reproductive'] = (
    (df_proc['gender'] == 'Female') & (df_proc['age'] <= 50)).astype(int)

print(f"Added features. New shape: {df_proc.shape}")

## 3. Encoding

Converting all text columns to numbers.

In [ ]:
# ordinal features - these have a natural order
ordinal_maps = {
    'exercise_level': {'Sedentary': 0, 'Light': 1, 'Moderate': 2, 'Active': 3},
    'eat_processed_food': {'Never': 0, 'Rarely': 1, 'Moderate': 2, 'Heavy': 3},
    'shortness_of_breath': {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3},
    'difficulty_falling_asleep': {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3},
    'frequent_headaches': {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3},
    'digestive_issues': {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3},
    'social_interaction_level': {'Low': 0, 'Moderate': 1, 'High': 2},
    'metabolism_type': {'Slow': 0, 'Normal': 1, 'Fast': 2},
    'perceived_appetite': {'Low': 0, 'Normal': 1, 'Excessive': 2},
    'sun_exposure': {'Low': 0, 'Moderate': 1, 'High': 2},
    'alcohol_consumption': {'Never': 0, 'Rarely': 1, 'Moderate': 2, 'Heavy': 3, 'Former': 4},
    'menstrual_regularity': {'N/A': -1, 'Regular': 0, 'Irregular': 1, 'Very Irregular': 2},
}

for col, mapping in ordinal_maps.items():
    df_proc[col] = df_proc[col].map(mapping)

# binary features - yes/no stuff
binary_maps = {
    'eat_fruits_daily': {'Yes': 1, 'No': 0},
    'eat_veggies_daily': {'Yes': 1, 'No': 0},
    'family_history_diabetes': {'Yes': 1, 'No': 0},
    'family_history_heart_disease': {'Yes': 1, 'No': 0},
    'family_history_hypertension': {'Yes': 1, 'No': 0},
    'family_history_obesity': {'Yes': 1, 'No': 0},
    'has_asthma': {'Yes': 1, 'No': 0},
    'has_thyroid': {'Yes': 1, 'No': 0},
    'has_allergies': {'Yes': 1, 'No': 0},
    'frequent_urination': {'Yes': 1, 'No': 0},
    'slow_wound_healing': {'Yes': 1, 'No': 0},
    'numbness_tingling': {'Yes': 1, 'No': 0},
    'family_history_pcos': {'Yes': 1, 'No': 0, 'N/A': -1},
    'smoking_status': {'Never': 0, 'Former': 1, 'Current': 2},
}

for col, mapping in binary_maps.items():
    df_proc[col] = df_proc[col].map(mapping)

# one-hot for nominal features (no natural ordering)
df_proc = pd.get_dummies(df_proc, columns=['gender', 'diet_type', 'employment_status', 'work_type'],
                         drop_first=True, dtype=int)

print(f"Shape after encoding: {df_proc.shape}")

## 4. Train-test split

In [ ]:
# build feature matrix
drop_cols = risk_score_cols + risk_level_cols + ['bmi_category', 'exercise_num']
X = df_proc.drop(columns=[c for c in drop_cols if c in df_proc.columns])

# make sure everything is numeric
assert X.select_dtypes(include='object').shape[1] == 0, "Still have text columns!"
print(f"Feature matrix: {X.shape}")

# encode targets
targets = [
    ('diabetes_risk_level', 'Diabetes'),
    ('heart_disease_risk_level', 'Heart Disease'),
    ('hypertension_risk_level', 'Hypertension'),
    ('obesity_risk_level', 'Obesity'),
    ('mental_health_risk_level', 'Mental Health'),
    ('respiratory_risk_level', 'Respiratory'),
    ('general_physical_health_level', 'General Physical Health'),
]

target_encoders = {}
Y = {}
for col, name in targets:
    le = LabelEncoder()
    Y[col] = le.fit_transform(df[col])
    target_encoders[col] = le
    print(f"  {name}: {list(le.classes_)}")

In [ ]:
# 80/20 split
X_train, X_test, idx_train, idx_test = train_test_split(
    X, np.arange(len(X)), test_size=0.2, random_state=42,
    stratify=Y['diabetes_risk_level']
)

# scale features
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_sc = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

Y_train = {col: Y[col][idx_train] for col in Y}
Y_test = {col: Y[col][idx_test] for col in Y}

print(f"Train: {X_train_sc.shape[0]:,}  |  Test: {X_test_sc.shape[0]:,}")

## 5. Model comparison

Trying a few models on diabetes risk to see which works best.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Extra Trees': ExtraTreesClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42),
}

y_tr = Y_train['diabetes_risk_level']
y_te = Y_test['diabetes_risk_level']

print(f"{'Model':<25s}  {'Train':>7s}  {'Test':>7s}  {'F1':>7s}")
print("-" * 52)

results = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_sc, y_tr)
    y_pred = model.predict(X_test_sc)
    
    train_acc = model.score(X_train_sc, y_tr)
    test_acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred, average='weighted')
    elapsed = time.time() - t0
    
    results[name] = {'model': model, 'test_acc': test_acc, 'f1': f1}
    print(f"  {name:<23s}  {train_acc:.4f}  {test_acc:.4f}  {f1:.4f}  ({elapsed:.1f}s)")

best = max(results, key=lambda k: results[k]['test_acc'])
print(f"\nBest: {best}")

## 6. Full evaluation - all 7 targets

Training Gradient Boosting on every target.

In [ ]:
all_results = {}

print(f"{'Target':<30s}  {'Acc':>6s}  {'F1':>6s}  {'AUC':>6s}")
print("-" * 55)

for col, name in targets:
    y_tr = Y_train[col]
    y_te = Y_test[col]
    n_classes = len(np.unique(y_tr))
    
    gb = GradientBoostingClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                     min_samples_leaf=10, subsample=0.8, random_state=42)
    gb.fit(X_train_sc, y_tr)
    
    y_pred = gb.predict(X_test_sc)
    y_proba = gb.predict_proba(X_test_sc)
    
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred, average='weighted')
    
    y_te_bin = label_binarize(y_te, classes=list(range(n_classes)))
    if n_classes == 2:
        y_te_bin = np.column_stack([1 - y_te_bin, y_te_bin])
    roc = roc_auc_score(y_te_bin, y_proba, multi_class='ovr', average='weighted')
    
    all_results[col] = {
        'name': name, 'model': gb, 'y_pred': y_pred, 'y_proba': y_proba,
        'acc': acc, 'f1': f1, 'roc_auc': roc, 'n_classes': n_classes
    }
    print(f"  {name:<28s}  {acc:.4f}  {f1:.4f}  {roc:.4f}")

In [ ]:
# classification reports for each target
for col, name in targets:
    r = all_results[col]
    le = target_encoders[col]
    print(f"\n{'='*55}")
    print(f"  {name}")
    print('='*55)
    print(classification_report(Y_test[col], r['y_pred'], target_names=le.classes_, digits=4))

### Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for i, (col, name) in enumerate(targets):
    ax = axes.flatten()[i]
    r = all_results[col]
    le = target_encoders[col]
    cm = confusion_matrix(Y_test[col], r['y_pred'])
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes_)
    ax.set_title(f"{name} (acc={r['acc']:.3f})", fontsize=9)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

axes.flatten()[-1].axis('off')
plt.suptitle("Confusion matrices (% of actual)", fontweight='bold')
plt.tight_layout()
plt.show()

### ROC curves

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for i, (col, name) in enumerate(targets):
    ax = axes.flatten()[i]
    r = all_results[col]
    le = target_encoders[col]
    n_classes = r['n_classes']
    y_te_bin = label_binarize(Y_test[col], classes=list(range(n_classes)))
    if n_classes == 2:
        y_te_bin = np.column_stack([1 - y_te_bin, y_te_bin])
    
    for cls in range(n_classes):
        fpr, tpr, _ = roc_curve(y_te_bin[:, cls], r['y_proba'][:, cls])
        ax.plot(fpr, tpr, label=f'{le.classes_[cls]} ({auc(fpr, tpr):.2f})')
    
    ax.plot([0,1], [0,1], 'k--', alpha=0.3)
    ax.set_title(f"{name} (AUC={r['roc_auc']:.3f})", fontsize=9)
    ax.legend(fontsize=7)

axes.flatten()[-1].axis('off')
plt.suptitle("ROC curves", fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Feature importance

What features matter most for each risk?

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 12))
for i, (col, name) in enumerate(targets):
    ax = axes.flatten()[i]
    imp = pd.Series(all_results[col]['model'].feature_importances_,
                    index=X.columns).sort_values()
    imp.tail(15).plot.barh(ax=ax, alpha=0.7)
    ax.set_title(name, fontsize=9)
    ax.tick_params(axis='y', labelsize=7)
axes.flatten()[-1].axis('off')
plt.suptitle("Top 15 features per target", fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# heatmap of importance across all targets
imp_df = pd.DataFrame()
for col, name in targets:
    imp_df[name] = pd.Series(all_results[col]['model'].feature_importances_, index=X.columns)

top20 = imp_df.mean(axis=1).sort_values(ascending=False).head(20).index

plt.figure(figsize=(11, 9))
sns.heatmap(imp_df.loc[top20], annot=True, fmt='.3f', cmap='YlOrRd')
plt.title("Feature importance across all targets (top 20)")
plt.tight_layout()
plt.show()

## 8. Save models and pipeline

In [ ]:
os.makedirs("saved_models", exist_ok=True)

# save each model separately
for col, name in targets:
    path = f"saved_models/{col}_model.pkl"
    with open(path, 'wb') as f:
        pickle.dump(all_results[col]['model'], f)
    print(f"  saved {path}")

# also save all models in one file for convenience
all_models = {col: all_results[col]['model'] for col, _ in targets}
with open("saved_models/all_models.pkl", 'wb') as f:
    pickle.dump(all_models, f)
print("  saved saved_models/all_models.pkl")

In [ ]:
# save the preprocessing stuff (scaler, encoders, mappings)
# need all of this to process new data the same way during inference
pipeline = {
    'scaler': scaler,
    'target_encoders': target_encoders,
    'feature_columns': list(X.columns),
    'ordinal_maps': ordinal_maps,
    'binary_maps': binary_maps,
    'target_configs': targets,
}

with open("saved_models/preprocessing_pipeline.pkl", 'wb') as f:
    pickle.dump(pipeline, f)
print("saved saved_models/preprocessing_pipeline.pkl")

In [ ]:
# save metrics as json (useful for the app dashboard)
metrics = {}
for col, name in targets:
    r = all_results[col]
    metrics[col] = {
        'name': name,
        'accuracy': round(r['acc'], 4),
        'f1_weighted': round(r['f1'], 4),
        'roc_auc': round(r['roc_auc'], 4),
    }

with open("saved_models/model_metrics.json", 'w') as f:
    json.dump(metrics, f, indent=2)
print("saved saved_models/model_metrics.json")

In [ ]:
# save preprocessed dataset too
export_df = X.copy()
for col, _ in targets:
    export_df[col] = Y[col]
for c in risk_score_cols:
    export_df[c] = df[c].values

export_df.to_csv("preprocessed_health_dataset.csv", index=False)
print(f"saved preprocessed_health_dataset.csv ({export_df.shape})")

In [ ]:
# quick check everything saved properly
print("\nAll saved files:")
for root, dirs, files in os.walk("saved_models"):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f"  {path:45s}  {size/1024:.0f} KB")
print(f"  {'preprocessed_health_dataset.csv':45s}  {os.path.getsize('preprocessed_health_dataset.csv')/1024/1024:.1f} MB")